# 서브워드 토큰화 실습 (BPE, BBPE, WordPiece)
이 노트북에서는 Hugging Face의 `datasets`, `transformers`, `tokenizers` 라이브러리와 표준 데이터를 활용하여 세 가지 주요 서브워드 토큰화 알고리즘을 실습합니다.

## 1. 환경 설정 및 데이터 준비
먼저 필요한 라이브러리를 설치하고, Hugging Face Hub에서 표준 데이터셋을 불러옵니다. 여기서는 영어 텍스트인 `wikitext` 데이터셋을 사용합니다.

In [1]:
# 필요 라이브러리 설치 (최초 1회 실행)
# !pip install datasets transformers tokenizers

from datasets import load_dataset

# Hugging Face 표준 데이터셋 로드 (Wikitext-2)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

print(f"데이터 개수: {len(dataset)}")
print(f"샘플 데이터: {dataset[10]['text']}")

README.md: 0.00B [00:00, ?B/s]

C:\Users\정세윤\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\정세윤\.cache\huggingface\hub\datasets--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

데이터 개수: 36718
샘플 데이터:  The game 's battle system , the BliTZ system , is carried over directly from Valkyira Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player moves the character around the battlefield in third @-@ person . A character can only act once per @-@ turn , but characters can be granted multiple turns at the expense of other characters ' turns . Each character has a field and distance of movement limited by their Action Gauge . Up to nine characters can be assigned to a single mission . During gameplay , characters will call out if something happens to them , such as their health points ( HP ) getting low or being knocked out by enemy attacks . Each character has specific " Potentials " , skills unique to each character . They are divided into " Personal Potential " , which are innate skills that remain unaltered unless otherwise dictated by the story and can either help

## 2. BPE (Byte Pair Encoding) 직접 학습시키기
Hugging Face의 `tokenizers` 라이브러리를 사용하여 로드한 표준 데이터셋으로 BPE 토크나이저를 직접 학습시켜 봅니다. 미등록 단어(OOV)를 어떻게 처리하는지 관찰합니다.

In [2]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 모델 초기화 및 Pre-tokenizer 세팅
tokenizer_bpe = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer_bpe.pre_tokenizer = Whitespace()

# vocab_size를 작게 주어 더 잘게 쪼개지도록 유도
trainer = BpeTrainer(special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"], vocab_size=5000)

# 데이터를 배치로 넘겨주기 위한 제너레이터
def batch_iterator(batch_size=1000):
    for i in range(0, len(dataset), batch_size):
        yield dataset[i : i + batch_size]["text"]

print("BPE 학습 시작...")
tokenizer_bpe.train_from_iterator(batch_iterator(), trainer=trainer)
print("학습 완료!")

sample_text = "Hugging Face is creating a tool that democratizes AI."
encoded_bpe = tokenizer_bpe.encode(sample_text)
print(f"BPE 토큰화 결과: {encoded_bpe.tokens}")

BPE 학습 시작...


학습 완료!
BPE 토큰화 결과: ['Hu', 'gg', 'ing', 'F', 'ace', 'is', 'cre', 'ating', 'a', 'to', 'ol', 'that', 'de', 'mo', 'cr', 'at', 'iz', 'es', 'AI', '.']


## 3. BBPE (Byte-Level BPE) - 사전학습 모델 활용
BBPE는 모든 문자를 바이트 단위로 분해하여 처리하므로 OOV(Out-Of-Vocabulary) 문제가 없습니다. 대표적인 모델인 GPT-2의 토크나이저를 사용해 봅니다.

In [3]:
from transformers import AutoTokenizer

# GPT-2 모델의 토크나이저 로드 (BBPE 방식)
tokenizer_bbpe = AutoTokenizer.from_pretrained("gpt2")

encoded_bbpe_eng = tokenizer_bbpe.tokenize(sample_text)
print(f"BBPE (영어): {encoded_bbpe_eng}")
# 'Ġ' 기호는 공백을 의미합니다.

korean_text = "허깅페이스(Hugging Face)로 자연어 처리를 배웁니다! 🚀"
encoded_bbpe_kor = tokenizer_bbpe.tokenize(korean_text)
print(f"BBPE (한국어/이모지): {encoded_bbpe_kor}")
# 한국어나 이모지가 바이트 단위로 쪼개진 것을 확인합니다.

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\정세윤\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\정세윤\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


BBPE (영어): ['Hug', 'ging', 'ĠFace', 'Ġis', 'Ġcreating', 'Ġa', 'Ġtool', 'Ġthat', 'Ġdemocrat', 'izes', 'ĠAI', '.']
BBPE (한국어/이모지): ['í', 'Ĺ', 'Ī', 'ê', '¹', 'ħ', 'í', 'İ', 'ĺ', 'ìĿ', '´', 'ì', 'Ĭ', '¤', '(', 'Hug', 'ging', 'ĠFace', ')', 'ë', '¡', 'ľ', 'Ġì', 'ŀ', 'Ĳ', 'ì', 'Ĺ', '°', 'ì', 'ĸ', '´', 'Ġì', '²', 'ĺ', 'ë', '¦', '¬', 'ë', '¥', '¼', 'Ġë', '°', '°', 'ì', 'Ľ', 'ģ', 'ëĭ', 'Ī', 'ëĭ', '¤', '!', 'ĠðŁ', 'ļ', 'Ģ']


## 4. WordPiece - 사전학습 모델 활용
WordPiece는 빈도수 대신 확률을 기반으로 병합합니다. 대표적인 모델인 BERT의 토크나이저를 사용해 봅니다.

In [4]:
# BERT 모델의 토크나이저 로드 (WordPiece 방식)
tokenizer_wordpiece = AutoTokenizer.from_pretrained("bert-base-uncased")

encoded_wp = tokenizer_wordpiece.tokenize(sample_text.lower())
print(f"WordPiece 토큰화 결과: {encoded_wp}")
# 서브워드 앞에는 '##' 기호가 붙습니다.

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

C:\Users\정세윤\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\정세윤\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


WordPiece 토큰화 결과: ['hugging', 'face', 'is', 'creating', 'a', 'tool', 'that', 'democrat', '##izes', 'ai', '.']


## 5. 종합 비교 분석
동일한 문장을 세 가지 방식에 모두 통과시켜 비교합니다.

In [5]:
test_sentence = "Unbelievably, tokenization is fascinating!"

print("1. BPE:")
print(tokenizer_bpe.encode(test_sentence).tokens)
print("-" * 50)

print("2. BBPE (GPT-2):")
print(tokenizer_bbpe.tokenize(test_sentence))
print("-" * 50)

print("3. WordPiece (BERT):")
print(tokenizer_wordpiece.tokenize(test_sentence.lower()))

1. BPE:
['Un', 'beli', 'ev', 'ably', ',', 'to', 'ken', 'ization', 'is', 'f', 'as', 'c', 'in', 'ating', '!']
--------------------------------------------------
2. BBPE (GPT-2):
['Un', 'bel', 'iev', 'ably', ',', 'Ġtoken', 'ization', 'Ġis', 'Ġfascinating', '!']
--------------------------------------------------
3. WordPiece (BERT):
['un', '##bel', '##ie', '##va', '##bly', ',', 'token', '##ization', 'is', 'fascinating', '!']


## 6. 비용 분석 (Cost Analysis) 실습
동일한 정보량(글자 수)을 가진 영어와 한국어 문장을 각각 처리할 때, 모델별로 생성되는 토큰 수를 세어보고 API 예상 청구 비용의 차이를 계산해 봅니다.
이 실습은 `README.md`에서 언급된 다국어 서비스에서의 요금 폭탄 문제를 체감하기 위해 작성되었습니다.

In [6]:
# 토큰 1,000개당 API 비용을 임의로 $0.002로 가정
cost_per_1k_tokens = 0.002

# 동일한 의미를 가진 영어 문장과 한국어 문장
text_en = "Machine learning is fascinating and it changes the world."
text_ko = "기계 학습은 매력적이며 세상을 변화시킵니다."

print(f"[영어 텍스트 길이]: {len(text_en)} 글자")
print(f"[한국어 텍스트 길이]: {len(text_ko)} 글자\n")

# BPE (GPT-2) 결과
tokens_en_bbpe = tokenizer_bbpe.tokenize(text_en)
tokens_ko_bbpe = tokenizer_bbpe.tokenize(text_ko)

# WordPiece (BERT) 결과
tokens_en_wp = tokenizer_wordpiece.tokenize(text_en)
tokens_ko_wp = tokenizer_wordpiece.tokenize(text_ko)

def print_cost_analysis(model_name, tokens_en, tokens_ko):
    count_en = len(tokens_en)
    count_ko = len(tokens_ko)
    
    # 100만 번 API를 호출한다고 가정할 때의 비용 계산
    api_calls = 1000000
    cost_en = (count_en * api_calls / 1000) * cost_per_1k_tokens
    cost_ko = (count_ko * api_calls / 1000) * cost_per_1k_tokens
    
    print(f"==== {model_name} 토크나이저 비용 분석 ====")
    print(f" * 영어 토큰 수: {count_en} 개")
    print(f" * 한국어 토큰 수: {count_ko} 개 (영문 대비 {(count_ko/count_en):.1f}배)")
    print(f" * 100만 회 호출 시 영어 처리 비용: ${cost_en:,.2f}")
    print(f" * 100만 회 호출 시 한국어 처리 비용: ${cost_ko:,.2f}\n")

print_cost_analysis("GPT-2 (BBPE)", tokens_en_bbpe, tokens_ko_bbpe)
print_cost_analysis("BERT (WordPiece)", tokens_en_wp, tokens_ko_wp)

print("""[결론]
GPT-2(영문 위주 BBPE 모델)를 한국어에 그대로 사용하면 토큰이 잘게 쪼개져 
영어 처리 대비 약 3배~4배 이상의 API 비용이 청구될 수 있습니다. 
따라서 다국어 또는 한국어 서비스에는 이에 특화된 토크나이저(KoBERT, SentencePiece 등)를 사용하는 것이 경제적입니다.""")


[영어 텍스트 길이]: 57 글자
[한국어 텍스트 길이]: 24 글자

==== GPT-2 (BBPE) 토크나이저 비용 분석 ====
 * 영어 토큰 수: 10 개
 * 한국어 토큰 수: 53 개 (영문 대비 5.3배)
 * 100만 회 호출 시 영어 처리 비용: $20.00
 * 100만 회 호출 시 한국어 처리 비용: $106.00

==== BERT (WordPiece) 토크나이저 비용 분석 ====
 * 영어 토큰 수: 10 개
 * 한국어 토큰 수: 45 개 (영문 대비 4.5배)
 * 100만 회 호출 시 영어 처리 비용: $20.00
 * 100만 회 호출 시 한국어 처리 비용: $90.00

[결론]
GPT-2(영문 위주 BBPE 모델)를 한국어에 그대로 사용하면 토큰이 잘게 쪼개져 
영어 처리 대비 약 3배~4배 이상의 API 비용이 청구될 수 있습니다. 
따라서 다국어 또는 한국어 서비스에는 이에 특화된 토크나이저(KoBERT, SentencePiece 등)를 사용하는 것이 경제적입니다.


---
## 7. 한국어 특화 토크나이저 비용 증명 (KoGPT2 실습)
위의 6번 분석에서 확인한 비용 폭탄 문제를 해결하기 위해, 실제로 한국어에 특화된 토크나이저(SKT의 KoGPT2)를 로드하여 비교해 봅시다.

In [7]:
# KoGPT2 토크나이저 로드 (BBPE 기반)
tokenizer_ko = AutoTokenizer.from_pretrained("skt/kogpt2-base-v2")

tokens_ko_kogpt2 = tokenizer_ko.tokenize(text_ko)

print("==== KoGPT2 (한국어 특화) 토크나이저 비용 분석 ====")
print(f" * 원본 문장: {text_ko}")
print(f" * GPT-2(영문) 토큰 수: {len(tokens_ko_bbpe):2d} 개 -> {tokens_ko_bbpe[:5]}...")
print(f" * KoGPT2 토큰 수:      {len(tokens_ko_kogpt2):2d} 개 -> {tokens_ko_kogpt2[:5]}...")

cost_kogpt2 = (len(tokens_ko_kogpt2) * 1000000 / 1000) * cost_per_1k_tokens
cost_bbpe = (len(tokens_ko_bbpe) * 1000000 / 1000) * cost_per_1k_tokens
print(f"\n * 100만 회 호출 시 GPT-2 비용: ${cost_bbpe:,.2f}")
print(f" * 100만 회 호출 시 KoGPT2 비용: ${cost_kogpt2:,.2f}")
print(f" -> 약 {(cost_bbpe/cost_kogpt2):.1f}배의 비용 절감 효과가 발생합니다! 이것이 독자 파운데이션 모델(독파모)들이 자체 토크나이저를 구축하는 이유입니다.")

config.json: 0.00B [00:00, ?B/s]

C:\Users\정세윤\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\정세윤\.cache\huggingface\hub\models--skt--kogpt2-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json: 0.00B [00:00, ?B/s]

==== KoGPT2 (한국어 특화) 토크나이저 비용 분석 ====
 * 원본 문장: 기계 학습은 매력적이며 세상을 변화시킵니다.
 * GPT-2(영문) 토큰 수: 53 개 -> ['ê', '¸', '°', 'ê', '³']...
 * KoGPT2 토큰 수:      24 개 -> ['ê', '°', 'ê', 'í', 'ì']...

 * 100만 회 호출 시 GPT-2 비용: $106.00
 * 100만 회 호출 시 KoGPT2 비용: $48.00
 -> 약 2.2배의 비용 절감 효과가 발생합니다! 이것이 독자 파운데이션 모델(독파모)들이 자체 토크나이저를 구축하는 이유입니다.


---
## 8. Vocab Size 튜닝에 따른 정보 압축률 비교
BPE 모델을 학습할 때 어휘 사전의 크기(`vocab_size`)는 성능과 속도에 결정적인 영향을 미칩니다. 사이즈를 다르게 주었을 때 시퀀스 길이가 어떻게 변하는지 확인합니다.

In [8]:
def train_and_evaluate_bpe(v_size):
    tz = Tokenizer(BPE(unk_token="[UNK]"))
    tz.pre_tokenizer = Whitespace()
    tr = BpeTrainer(special_tokens=["[UNK]"], vocab_size=v_size)
    
    # 빠른 학습을 위해 5000개 데이터만 사용
    subset_data = dataset[:5000]["text"]
    tz.train_from_iterator(subset_data, trainer=tr)
    
    test_str = "Tokenization algorithms dictate how language models perceive text."
    encoded = tz.encode(test_str)
    print(f"Vocab Size: {v_size:5d} | Token Length: {len(encoded.tokens):2d} | Tokens: {encoded.tokens}")

print("==== Vocab Size에 따른 토큰화 길이 비교 ====")
train_and_evaluate_bpe(1000)
train_and_evaluate_bpe(5000)
train_and_evaluate_bpe(10000)

print("\n[인사이트]: Vocab Size가 커질수록 긴 단어가 통째로 사전에 등재되어 시퀀스 길이는 짧아집니다. ")
print("하지만 사전 크기가 너무 커지면 모델의 파라미터 수(Embedding Layer)가 폭발적으로 증가하는 트레이드오프가 존재합니다.")

==== Vocab Size에 따른 토큰화 길이 비교 ====
Vocab Size:  1000 | Token Length: 29 | Tokens: ['T', 'ok', 'en', 'iz', 'ation', 'al', 'g', 'or', 'ith', 'm', 's', 'd', 'ict', 'ate', 'how', 'l', 'ang', 'u', 'age', 'm', 'od', 'el', 's', 'per', 'ce', 'ive', 't', 'ext', '.']


Vocab Size:  5000 | Token Length: 20 | Tokens: ['T', 'ok', 'en', 'ization', 'al', 'g', 'or', 'ith', 'ms', 'dict', 'ate', 'how', 'language', 'mod', 'els', 'per', 'ce', 'ive', 'text', '.']


Vocab Size: 10000 | Token Length: 20 | Tokens: ['T', 'ok', 'en', 'ization', 'al', 'g', 'or', 'ith', 'ms', 'dict', 'ate', 'how', 'language', 'mod', 'els', 'per', 'ce', 'ive', 'text', '.']

[인사이트]: Vocab Size가 커질수록 긴 단어가 통째로 사전에 등재되어 시퀀스 길이는 짧아집니다. 
하지만 사전 크기가 너무 커지면 모델의 파라미터 수(Embedding Layer)가 폭발적으로 증가하는 트레이드오프가 존재합니다.
